In [24]:
from movie_recommender.models.popularity import PopularityRecommender
from movie_recommender.evaluation.metrics import hit_rate_k, ndcg_k
from movie_recommender.data.split import temporal_positive_split
import pandas as pd
from pathlib import Path

In [4]:
rating = pd.read_csv("../dataset/ml-latest-small/ratings.csv")

In [7]:
train, validation, test = temporal_positive_split(rating)

In [8]:
model = PopularityRecommender()
model.fit(train)

In [9]:
relevant_by_user = validation.groupby('userId')['movieId'].apply(set).to_dict()

In [10]:
print(relevant_by_user)

{1: {2012}, 2: {122882}, 3: {7899}, 4: {4896}, 5: {247}, 6: {1061}, 7: {48783}, 8: {11}, 9: {5965}, 10: {7154}, 11: {1584}, 12: {1721}, 13: {3827}, 14: {376}, 15: {84954}, 16: {91529}, 17: {111}, 18: {177593}, 19: {913}, 20: {1088}, 21: {47997}, 22: {3578}, 23: {6}, 24: {5791}, 25: {177593}, 26: {47}, 27: {2407}, 28: {316}, 29: {104841}, 30: {110}, 31: {1302}, 32: {1084}, 33: {919}, 34: {2443}, 35: {590}, 36: {3347}, 37: {1073}, 38: {246}, 39: {2391}, 40: {121}, 41: {6763}, 42: {2805}, 43: {1064}, 44: {889}, 45: {53121}, 46: {236}, 47: {1097}, 48: {5882}, 49: {79132}, 50: {1251}, 51: {60487}, 52: {50794}, 53: {1049}, 55: {4011}, 56: {69}, 57: {494}, 58: {708}, 59: {2726}, 60: {6016}, 61: {7361}, 62: {46976}, 63: {89904}, 64: {1345}, 65: {58559}, 66: {523}, 67: {99114}, 68: {117529}, 69: {4027}, 70: {1303}, 71: {1370}, 72: {648}, 73: {62434}, 74: {57147}, 75: {1240}, 76: {2318}, 77: {5378}, 78: {1270}, 79: {1127}, 80: {593}, 81: {589}, 82: {7325}, 83: {4975}, 84: {1380}, 85: {1237}, 86:

## since popularity model can recommend movies in train dataset, so exclude user with relevant movies not seen in train

In [11]:
train_catalog = set(train['movieId'])

In [12]:

evaluable_relevant_by_user = {user_id : relevant_movies 
                              for user_id, relevant_movies in relevant_by_user.items() 
                              if relevant_movies.issubset(train_catalog)}

In [13]:
excluded_users = (
    len(relevant_by_user)
    - len(evaluable_relevant_by_user)
)

In [15]:
print("Validation users:", len(relevant_by_user))
print("Evaluable users:", len(evaluable_relevant_by_user))
print("Excluded users:", excluded_users)

Validation users: 603
Evaluable users: 566
Excluded users: 37


# Evaluation of Popularity model on relevant movie from validation

In [17]:
evaluation_rows = []
for user_id, relevant_movies in evaluable_relevant_by_user.items():
    seen_movies = set(train.loc[train['userId']==user_id, 'movieId'])

    recommendations = model.recommend(seen_movies=seen_movies, k=10)

    hit_rate = hit_rate_k(recommendations, relevant_movies, k=10)
    ndcg = ndcg_k(recommendations, relevant_movies, k = 10)

    evaluation_rows.append({
        'userId': user_id,
        'relevant_movie': next(iter(relevant_movies)),
        'recommendations': recommendations,
        'hit_rate@10': hit_rate,
        'ndcg@10': ndcg
        })


In [20]:
per_user_results = pd.DataFrame(evaluation_rows)

In [21]:
per_user_results.head()

,userId,relevant_movie,recommendations,hit_rate@10,ndcg@10
0,1,2012,"[318, 296, 858, 589, 4993, 7153, 5952, 150, 27...",0.0,0.0
1,2,122882,"[318, 356, 296, 2571, 593, 260, 2959, 527, 119...",0.0,0.0
2,4,4896,"[318, 356, 296, 2571, 2959, 527, 110, 50, 1198...",0.0,0.0
3,5,247,"[318, 356, 2571, 593, 260, 2959, 1196, 1198, 8...",0.0,0.0
4,6,1061,"[296, 2571, 260, 2959, 527, 1196, 50, 1198, 85...",0.0,0.0


In [23]:
popularity_results = {
    "model": "popularity",
    "k": 10,
    "users_evaluated": len(per_user_results),
    "hit_rate": float(per_user_results["hit_rate@10"].mean()),
    "ndcg": float(per_user_results["ndcg@10"].mean()),
}

print(popularity_results)

{'model': 'popularity', 'k': 10, 'users_evaluated': 566, 'hit_rate': 0.045936395759717315, 'ndcg': 0.01998717919735628}


In [28]:
validation_results_path = Path('../reports/tables/validation_results.csv')
validation_results_path.parent.mkdir(parents=True, exist_ok=True)

validation_result = pd.DataFrame([popularity_results])
validation_result.to_csv(validation_results_path)